In [4]:
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


# -----------------------------
# LOAD DATA
# -----------------------------

data = pd.read_csv("Data.csv")

# combine subject + description
data["text"] = data["Ticket Subject"] + " " + data["Ticket Description"]


# -----------------------------
# NLP PREPROCESSING SETUP
# -----------------------------

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess(text):

    text = text.lower()

    tokens = word_tokenize(text)

    cleaned_tokens = []

    for word in tokens:
        if word not in stop_words and word not in string.punctuation:
            lemma = lemmatizer.lemmatize(word)
            cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)


# clean the text
data["clean_text"] = data["text"].apply(preprocess)


# -----------------------------
# ENCODE TARGET VARIABLES
# -----------------------------

category_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

data["category_encoded"] = category_encoder.fit_transform(data["Ticket Type"])
data["priority_encoded"] = priority_encoder.fit_transform(data["Ticket Priority"])


# -----------------------------
# TEXT → NUMERIC FEATURES
# -----------------------------

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(data["clean_text"])

y_category = data["category_encoded"]
y_priority = data["priority_encoded"]


# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------

X_train, X_test, yc_train, yc_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42
)

_, _, yp_train, yp_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)


# -----------------------------
# TRAIN CATEGORY MODEL
# -----------------------------

category_model = LogisticRegression(max_iter=200)

category_model.fit(X_train, yc_train)


# -----------------------------
# TRAIN PRIORITY MODEL
# -----------------------------

priority_model = LogisticRegression(max_iter=200)

priority_model.fit(X_train, yp_train)


# -----------------------------
# EVALUATE CATEGORY MODEL
# -----------------------------

category_pred = category_model.predict(X_test)

# print("\nCategory Model Performance")
# print("----------------------------")

# print(classification_report(yc_test, category_pred))


# -----------------------------
# EVALUATE PRIORITY MODEL
# -----------------------------

priority_pred = priority_model.predict(X_test)

print("\nPriority Model Performance")
print("----------------------------")

print(classification_report(yp_test, priority_pred))


# -----------------------------
# USER INPUT PREDICTION
# -----------------------------

ticket = input("\nEnter support ticket: ")

clean_ticket = preprocess(ticket)

ticket_vec = vectorizer.transform([clean_ticket])


# predict category
category_prediction = category_model.predict(ticket_vec)

category = category_encoder.inverse_transform(category_prediction)


# predict priority
priority_prediction = priority_model.predict(ticket_vec)

priority = priority_encoder.inverse_transform(priority_prediction)


print("\nPredicted Results")
print("----------------------")

print("Category :", category[0])
print("Priority :", priority[0])






Category Model Performance
----------------------------

Priority Model Performance
----------------------------
              precision    recall  f1-score   support

           0       0.25      0.26      0.25       411
           1       0.24      0.25      0.24       409
           2       0.23      0.21      0.22       415
           3       0.29      0.29      0.29       459

    accuracy                           0.25      1694
   macro avg       0.25      0.25      0.25      1694
weighted avg       0.25      0.25      0.25      1694


Predicted Results
----------------------
Category : Technical issue
Priority : Low
